# Notebook 5: Hybrid LSTM-SNP (Fuzzy Feature Aug + Fuzzy Gates)

**Dataset**: Monthly Milk Production

## Description
This notebook implements the **hybrid model** that combines:
1. **Fuzzy Feature Augmentation** (from Notebook 2): Input is augmented with fuzzy-derived features
2. **Fuzzy Gate Replacement** (from Notebook 3): Gates use fuzzy inference instead of sigmoid

This represents the most complex integration of fuzzy logic with LSTM-SNP. Gradient clipping 
(norm=1.0) and careful weight initialization are applied for training stability.

The interaction between input-level and gate-level fuzzy reasoning is documented and analyzed.

## Theory: Hybrid Fuzzy LSTM-SNP

This model combines two fuzzy integration strategies:

### 1. Input-Level: Fuzzy Feature Augmentation
Same as Notebook 2:
- Fuzzy inference on $x(t)$ and $x(t-1)$
- Augmented input: $x'(t) = [x(t), y_{fuzzy}]$
- Input dimension becomes 2

### 2. Gate-Level: Fuzzy Gate Replacement
Same as Notebook 3:
- Gates r, c, o computed via fuzzy inference
- Fixed Gaussian MFs, trainable consequent parameters
- Generation gate a retains tanh

### Interaction Between Levels
- The fuzzy-augmented input provides richer information to the cell
- The fuzzy gates process this enriched input with adaptive, rule-based gating
- This creates a two-level fuzzy reasoning pipeline

### Stability Measures
- Gradient clipping (norm ≤ 1.0)
- Careful weight initialization
- Fuzzy weights are clipped to valid ranges

## Model Architecture & Implementation

In [ ]:
# ============================================================
# ALL IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

### Fuzzy Inference System (NumPy — for preprocessing)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Fuzzy Inference System (NumPy — for preprocessing)
#
# Fixed Gaussian membership functions:
#   μ_low(x)  = exp(-(x - (-1))² / (2·0.5²))
#   μ_high(x) = exp(-(x - (+1))² / (2·0.5²))
#
# 4 Takagi-Sugeno rules with fixed consequent parameters:
#   IF x(t) is low  AND x(t-1) is low  → y₁ = 0.5·x(t) + 0.5·x(t-1)
#   IF x(t) is low  AND x(t-1) is high → y₂ = 0.7·x(t) + 0.3·x(t-1) - 0.1
#   IF x(t) is high AND x(t-1) is low  → y₃ = 0.3·x(t) + 0.7·x(t-1) + 0.1
#   IF x(t) is high AND x(t-1) is high → y₄ = 0.5·x(t) + 0.5·x(t-1)
#
# Output: y = Σ(wᵢ·yᵢ) / Σ(wᵢ)
# ============================================================

def gaussian_mf(x, center, sigma=0.5):
    """Fixed Gaussian membership function."""
    return np.exp(-(x - center)**2 / (2 * sigma**2))

def fuzzy_inference_numpy(x_t, x_tm1):
    """
    Compute fuzzy feature from x(t) and x(t-1).
    Uses fixed membership functions and fixed consequent parameters.
    """
    # Membership degrees
    mu_low_xt = gaussian_mf(x_t, center=-1.0)
    mu_high_xt = gaussian_mf(x_t, center=1.0)
    mu_low_xtm1 = gaussian_mf(x_tm1, center=-1.0)
    mu_high_xtm1 = gaussian_mf(x_tm1, center=1.0)

    # Rule firing strengths (product)
    w1 = mu_low_xt * mu_low_xtm1      # low-low
    w2 = mu_low_xt * mu_high_xtm1     # low-high
    w3 = mu_high_xt * mu_low_xtm1     # high-low
    w4 = mu_high_xt * mu_high_xtm1    # high-high

    # Consequent outputs (fixed linear functions)
    y1 = 0.5 * x_t + 0.5 * x_tm1
    y2 = 0.7 * x_t + 0.3 * x_tm1 - 0.1
    y3 = 0.3 * x_t + 0.7 * x_tm1 + 0.1
    y4 = 0.5 * x_t + 0.5 * x_tm1

    # Weighted average defuzzification
    numerator = w1 * y1 + w2 * y2 + w3 * y3 + w4 * y4
    denominator = w1 + w2 + w3 + w4 + 1e-8

    return numerator / denominator

### Fuzzy Gate LSTM-SNP Cell

In [ ]:
# ============================================================
# Fuzzy Gate LSTM-SNP Cell (PyTorch Implementation)
# Gates r, c, o use fuzzy inference instead of hard sigmoid.
# Generation gate 'a' keeps tanh (unchanged).
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class FuzzyGateType5(nn.Module):
    def __init__(self, hidden_size, sigma=0.5):
        super().__init__()
        self.hidden_size = hidden_size
        self.sigma = sigma
        self.mu_low = -1.0
        self.mu_high = 1.0
        
        # Rule 0 (Low)
        self.a0 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.b0 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.c0 = nn.Parameter(torch.zeros(hidden_size))
        
        # Rule 1 (High)
        self.a1 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.b1 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.c1 = nn.Parameter(torch.zeros(hidden_size))

    def _gaussian_mf(self, x, center):
        return torch.exp(-(x - center)**2 / (2.0 * self.sigma**2))

    def forward(self, z_gate, u_mean):
        w_low = self._gaussian_mf(z_gate, self.mu_low)
        w_high = self._gaussian_mf(z_gate, self.mu_high)
        
        y0 = self.a0 * z_gate + self.b0 * u_mean + self.c0
        y1 = self.a1 * z_gate + self.b1 * u_mean + self.c1
        
        numerator = w_low * y0 + w_high * y1
        denominator = w_low + w_high + 1e-8
        output = numerator / denominator
        return torch.clamp(output, 0.0, 1.0)


class FuzzyLSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell with fuzzy gate replacement.
    Gates r, c, o are computed via fuzzy inference.
    Gate a keeps tanh activation.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.W = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.U = nn.Linear(hidden_size, 4 * hidden_size, bias=True)
        
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.orthogonal_(self.U.weight)
        nn.init.zeros_(self.W.bias)
        nn.init.zeros_(self.U.bias)
        
        self.fuzzy_r = FuzzyGateType5(hidden_size)
        self.fuzzy_c = FuzzyGateType5(hidden_size)
        self.fuzzy_o = FuzzyGateType5(hidden_size)

    def forward(self, x, u_prev):
        z = self.W(x) + self.U(u_prev)
        
        z0 = z[:, :self.hidden_size]
        z1 = z[:, self.hidden_size:2*self.hidden_size]
        z2 = z[:, 2*self.hidden_size:3*self.hidden_size]
        z3 = z[:, 3*self.hidden_size:]
        
        # Mean of previous state for fuzzy rule input
        u_mean = u_prev.mean(dim=-1, keepdim=True).expand_as(u_prev)
        
        r = self.fuzzy_r(z0, u_mean)
        c = self.fuzzy_c(z1, u_mean)
        o = self.fuzzy_o(z2, u_mean)
        a = torch.tanh(z3)
        
        u = r * u_prev - c * a
        h = o * a
        
        return h, u


### Build Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Model Construction (PyTorch)
# ============================================================

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        # Select cell
        if 5 in [1, 2, 4]:
            self.cell = LSTMSNPCell(input_size, hidden_size)
        else:
            self.cell = FuzzyLSTMSNPCell(input_size, hidden_size)
            
        # Select output layer
        if 5 == 4:
            self.out = FuzzyOutputLayer(hidden_size)
        else:
            self.out = nn.Linear(hidden_size, 1)
            
        self.u = None
        
    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)
        
    def forward(self, x):
        # x is (batch, 1, input_size)
        if self.u is None or self.u.device != x.device:
            self.reset_states(x.size(0), x.device)
            
        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)
        
def build_model(input_dim, units):
    return RNNModel(input_dim, units)


In [ ]:
# Quick model check
model = build_model(input_dim=2, units=8)
print(model)

## Data Pipeline — Monthly Milk Production

In [ ]:
import os
for f in os.listdir('/kaggle/input/datasets/satabartosarkar123/monthly-milk-production-pounds-p-csv'):
    print(f)

In [ ]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    '/kaggle/input/datasets/satabartosarkar123/monthly-milk-production-pounds-p-csv/monthly-milk-production-pounds-p.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)

original_raw_values = series.values.flatten()
print(f"Base data shape: {original_raw_values.shape}")
print(f"First 5 values: {original_raw_values[:5]}")

In [ ]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

In [ ]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

In [ ]:
# ============================================================
# 4-5-6. Split / Scale / Reshape (inside noise loop below)
# ============================================================
# - Last 60 for test, rest for training, NO validation
# - MinMaxScaler(-1, 1)
# - Fuzzy augmentation: fuzzy_inference_numpy(x_t, x_tm1) -> input_dim=2
print("Split: last 60 for test, rest for training, no validation")
print("Fuzzy augmentation: input_dim=2")

## Training Loop

In [ ]:
# ============================================================
# Gaussian Noise Evaluation + 60-Run Experiment Protocol (PyTorch)
# ============================================================
import torch
import torch.nn as nn

s_x = np.std(original_raw_values)
noise_levels = [0.0, 0.005]
_molab_results = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")

for lam in noise_levels:
    sigma = lam * s_x
    print("\n" + "="*80)
    print(f"EVALUATING NOISE LEVEL: {lam*100:.1f}% (lambda={lam}, sigma={sigma:.6f})")
    print("="*80 + "\n")
    
    np.random.seed(42)
    torch.manual_seed(42)
    noise = np.random.normal(0, sigma, size=original_raw_values.shape)
    raw_values = original_raw_values + noise
    
    # 2. Difference
    diff_values = difference(raw_values, 1)
    # 3. Supervised (lag=1)
    supervised = timeseries_to_supervised(diff_values, 1)
    # 4. Split (Last 60 for test, NO validation)
    train, test = supervised[:-60], supervised[-60:]
    # 5. Scale
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train)
    train_scaled = scaler.transform(train)
    test_scaled = scaler.transform(test)
    
    # 6. Reshape with Fuzzy Feature Augmentation (input_dim=2)
    X_train_raw = train_scaled[:, 0:1]
    y_train = train_scaled[:, 1]
    
    X_train_fuzzy = np.zeros((X_train_raw.shape[0], 2))
    for i in range(X_train_raw.shape[0]):
        x_t = X_train_raw[i, 0]
        x_tm1 = X_train_raw[i-1, 0] if i > 0 else x_t
        y_fuzz = fuzzy_inference_numpy(x_t, x_tm1)
        X_train_fuzzy[i, 0] = x_t
        X_train_fuzzy[i, 1] = y_fuzz
    
    X_train = X_train_fuzzy.reshape((X_train_fuzzy.shape[0], 1, 2))
    
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    
    all_rmse = []
    all_mse = []
    all_nmse = []
    all_predictions = []
    all_losses = []
    
    for run in range(60):
        print(f'\n===== RUN {run+1}/60 [Noise {lam*100:.1f}%] =====')
        
        np.random.seed(run)
        torch.manual_seed(run)
        
        model = build_model(input_dim=2, units=8).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        
        # Pre-tensorize training data
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)
        
        run_losses = []
        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            
            epoch_loss = 0.0
            for i in range(n_samples):
                x_i = X_train_t[i:i+1]
                y_i = y_train_t[i:i+1]
                
                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                model.u = model.u.detach()
                epoch_loss += loss.item()
            
            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)
            if (epoch+1) % 10 == 0:
                print(f"  Epoch {epoch+1}/100, Loss: {avg_loss:.6f}")
        
        all_losses.append(run_losses)
        print(f'Training complete for run {run+1}')
        
        # Warm-up: condition hidden states on training data
        model.eval()
        with torch.no_grad():
            model.reset_states(1, device)
            prev_x = None
            for i in range(len(train_scaled)):
                x_t = train_scaled[i, 0]
                x_tm1 = prev_x if prev_x is not None else x_t
                y_fuzz = fuzzy_inference_numpy(x_t, x_tm1)
                X_aug = torch.tensor([x_t, y_fuzz], dtype=torch.float32).view(1, 1, 2).to(device)
                model(X_aug)
                prev_x = x_t
        
        # Test predictions (single-step) with fuzzy feature augmentation
        predictions = []
        model.eval()
        with torch.no_grad():
            x_tm1 = train_scaled[-1, 0]  # last training value
            for i in range(len(test_scaled)):
                x_t = test_scaled[i, 0]
                y_fuzz = fuzzy_inference_numpy(x_t, x_tm1)
                X_aug = torch.tensor([x_t, y_fuzz], dtype=torch.float32).view(1, 1, 2).to(device)
                yhat = model(X_aug).item()
                
                # Invert scaling
                X_raw = test_scaled[i, 0:-1]
                new_row = list(X_raw) + [yhat]
                array = np.array(new_row).reshape(1, len(new_row))
                inverted = scaler.inverse_transform(array)[0, -1]
                
                # Invert differencing
                inverted = inverted + raw_values[len(train) + i]
                predictions.append(inverted)
                
                x_tm1 = x_t  # update for next step
        
        # Compute metrics
        actual = raw_values[-len(predictions):]
        rmse = sqrt(mean_squared_error(actual, predictions))
        mse = mean_squared_error(actual, predictions)
        meanV = np.mean(actual)
        dominator = np.linalg.norm(np.array(predictions) - meanV, 2)
        nmse = mse / np.power(dominator, 2)
        
        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)
        
        print(f'Run {run+1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')
    
    # End of 60 runs for this noise level
    best_idx = np.argmin(all_rmse)
    
    print(f'\n===== STATS FOR NOISE {lam*100:.1f}% (60 runs) =====')
    print(f'RMSE: {np.mean(all_rmse):.6f} +/- {np.std(all_rmse):.6f}')
    print(f'MSE:  {np.mean(all_mse):.6f} +/- {np.std(all_mse):.6f}')
    print(f'NMSE: {np.mean(all_nmse):.10f} +/- {np.std(all_nmse):.10f}')
    print(f'Best run: {best_idx+1} (RMSE: {all_rmse[best_idx]:.6f})')
    
    _molab_results.append([lam, 
        np.mean(all_rmse), np.std(all_rmse),
        np.mean(all_mse), np.std(all_mse),
        np.mean(all_nmse), np.std(all_nmse),
        all_rmse[best_idx], all_predictions[best_idx], all_losses[best_idx], raw_values.copy()])


## Results

In [ ]:
# ============================================================
# Final Metrics Summary & Plots per Noise Level
# ============================================================
from tabulate import tabulate

for entry in _molab_results:
    lam = entry[0]
    mean_rmse, std_rmse = entry[1], entry[2]
    mean_mse, std_mse = entry[3], entry[4]
    mean_nmse, std_nmse = entry[5], entry[6]
    best_rmse_val = entry[7]
    best_predictions = entry[8]
    best_losses = entry[9]
    raw_values_for_plot = entry[10]
    
    print("\n" + "="*80)
    print(f"RESULTS FOR NOISE LEVEL: {lam*100:.1f}%")
    print("="*80)
    
    print(f'Mean RMSE: {mean_rmse:.6f} +/- {std_rmse:.6f}')
    print(f'Mean MSE:  {mean_mse:.6f} +/- {std_mse:.6f}')
    print(f'Mean NMSE: {mean_nmse:.10f} +/- {std_nmse:.10f}')
    print(f'Best RMSE: {best_rmse_val:.6f}')
    
    actual = raw_values_for_plot[-60:]
    
    plt.figure(figsize=(12, 5))
    plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
    plt.plot(best_predictions, label='Predicted (Best Run)', color='red',
             linewidth=1.5, linestyle='--')
    plt.title(f'Predictions vs Actual (Noise {lam*100:.1f}%)')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(best_losses, color='green', linewidth=1.0)
    plt.title(f'Training Loss (Best Run, Noise {lam*100:.1f}%)')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\n" + "="*80)
print("FINAL RESULTS TABLE ACROSS ALL NOISE LEVELS")
print("="*80)
df_res = pd.DataFrame(
    [[r[0], f'{r[1]:.6f} +/- {r[2]:.6f}', f'{r[3]:.6f} +/- {r[4]:.6f}', 
      f'{r[5]:.10f} +/- {r[6]:.10f}', f'{r[7]:.6f}'] for r in _molab_results],
    columns=['Noise Level', 'RMSE (mean+/-std)', 'MSE (mean+/-std)', 'NMSE (mean+/-std)', 'Best RMSE']
)
print(tabulate(df_res, headers='keys', tablefmt='github', showindex=False))


## Observations

### Gaussian Noise robustness on Monthly Milk Production

**Run the notebook to populate results.**